cleaning circuit file

In [0]:
circuits_df=spark.read.table("f1_catalog.bronze.circuits")

In [0]:
circuits_df = circuits_df.select(
    "circuit_id",
    "circuit_ref",
    "name",
    "location",
    "country",
    "latitude",
    "longitude",
    "altitude"
)

In [0]:
circuits_df=circuits_df.dropDuplicates(["circuit_id"])

In [0]:
from pyspark.sql.functions import current_timestamp

circuits_df=circuits_df.withColumn("updated_date", current_timestamp())

In [0]:
circuits_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.circuits")

Cleaning races file 

In [0]:
races_df=spark.read.table("f1_catalog.bronze.races")

In [0]:
races_df = races_df.select(
    "race_id",
    "race_year",
    "round",
    "circuit_id",
    "name",
    "date",
    "time"
)

In [0]:
races_df=races_df.drop("time")

In [0]:
races_df=races_df.dropDuplicates(["race_id"])

In [0]:
from pyspark.sql.functions import col, count, when

races_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in races_df.columns
]).display()

In [0]:
races_df = races_df.dropna(subset=["race_id", "circuit_id", "race_year"])

In [0]:
from pyspark.sql.functions import current_timestamp
races_df = races_df.withColumn("updated_date", current_timestamp())

In [0]:
races_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.races")

In [0]:
%sql
select * from f1_catalog.silver.races

Cleaning constructor file

In [0]:
constructors_df = spark.read.table("f1_catalog.bronze.constructors")

In [0]:
from pyspark.sql.functions import col, count, when

constructors_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in constructors_df.columns
]).display()

In [0]:
constructors_df = constructors_df.dropna(subset=["constructor_id"])

In [0]:
from pyspark.sql.functions import lit, coalesce

constructors_df = constructors_df \
    .withColumn("constructor_ref", coalesce(col("constructor_ref"), lit("unknown"))) \
    .withColumn("name", coalesce(col("name"), lit("unknown"))) \
    .withColumn("nationality", coalesce(col("nationality"), lit("unknown")))

In [0]:
constructors_df = constructors_df.select(
    "constructor_id",
    "constructor_ref",
    "name",
    "nationality"
)

In [0]:
constructors_df=constructors_df.dropDuplicates(["constructor_id"])

In [0]:
constructors_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.constructors")

In [0]:
%sql
select * from f1_catalog.silver.constructors

cleaning driver file

In [0]:
drivers_df = spark.read.table("f1_catalog.bronze.drivers")

In [0]:
drivers_df = drivers_df \
    .drop("url", "name") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("driverRef", "driver_ref")

In [0]:
drivers_df = drivers_df.dropna(subset=["driver_id"])

Create full name

In [0]:
from pyspark.sql.functions import concat, lit, col

drivers_df = drivers_df.withColumn(
    "full_name",
    concat(col("forename"), lit(" "), col("surname"))
)

In [0]:
drivers_df = drivers_df.select(
    "driver_id",
    "driver_ref",
    "code",
    "full_name",
    "dob",
    "nationality"
)

In [0]:
drivers_df=drivers_df.dropDuplicates(["driver_id"])

In [0]:
drivers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.drivers")

Cleaning result file

In [0]:
results_df=spark.read.table("f1_catalog.bronze.results")

In [0]:
from pyspark.sql.functions import col, when

results_df = results_df.replace("\\N", None)

In [0]:
results_df = results_df \
    .withColumnRenamed("resultId", "result_id") \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("positionOrder", "position_order")

In [0]:
results_df = results_df.dropna(subset=["result_id", "race_id", "driver_id"])

In [0]:
results_df = results_df.select(
    "result_id",
    "race_id",
    "driver_id",
    "constructor_id",
    "position",
    "points",
    "laps",
    "milliseconds",
    "fastestLap",
    "fastestLapTime",
    "fastestLapSpeed"
)

In [0]:
results_df = results_df \
    .withColumnRenamed("fastestLap", "fastest_lap") \
    .withColumnRenamed("fastestLapTime", "fastest_lap_time") \
    .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")

In [0]:
results_df = results_df.dropDuplicates(["result_id"])

In [0]:
%sql
drop table f1_catalog.silver.results

In [0]:
results_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.results")

In [0]:
%sql 
select * from f1_catalog.silver.results

### Cleaning pit_stop file

In [0]:
pit_stops_df = spark.read.table("f1_catalog.bronze.pit_stops")

In [0]:
pit_stops_df = pit_stops_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id")

In [0]:
pit_stops_df = pit_stops_df.dropna(subset=["race_Id", "driver_Id"])

In [0]:
pit_stops_df = pit_stops_df.select(
    "race_id",
    "driver_id",
    "stop",
    "lap",
    "time",
    "duration",
    "milliseconds"
)

In [0]:
pit_stops_df = pit_stops_df.dropDuplicates()

In [0]:
pit_stops_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.pit_stops")

Cleaning lap time

In [0]:
lap_times_df=spark.read.table("f1_catalog.bronze.lap_times")

In [0]:
lap_times_df.columns

In [0]:
lap_times_df = lap_times_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id")

In [0]:
lap_times_df = lap_times_df.dropna(subset=["race_id", "driver_id"])

In [0]:
lap_times_df = lap_times_df.select(
    "race_id",
    "driver_id",
    "lap",
    "position",
    "time",
    "milliseconds"
)

In [0]:
lap_times_df = lap_times_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import current_timestamp

lap_times_df = lap_times_df.withColumn("updated_date", current_timestamp())

In [0]:
lap_times_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.lap_times")

cleaning qualyfying file

In [0]:
lap_times_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.lap_times")

In [0]:
qualifying_df = spark.read.table("f1_catalog.bronze.qualifying")

In [0]:
qualifying_df.selectExpr([
    f"sum(case when {c} is null then 1 else 0 end) as {c}" 
    for c in qualifying_df.columns
]).display()

In [0]:
qualifying_df = qualifying_df \
    .withColumnRenamed("qualifyId", "qualify_id") \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("constructorId", "constructor_id")

In [0]:
qualifying_df = qualifying_df.select(
    "qualify_id",
    "race_id",
    "driver_id",
    "constructor_id",
    "number",
    "position",
    "q1",
    "q2",
    "q3"
)

In [0]:
qualifying_df = qualifying_df.dropDuplicates(["qualify_id"])

In [0]:
from pyspark.sql.functions import current_timestamp

qualifying_df = qualifying_df.withColumn("updated_date", current_timestamp())

In [0]:
qualifying_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("f1_catalog.silver.qualifying")